# Chatbot General usando LLM Destilado Local

Vamos a descargar un modelo en formato GGUF (Optimizado para CPU) y usarlo localmente usando la librería `llama-cpp-python` y `langchain`.

In [4]:
!pip install ipywidgets
from IPython.display import display, clear_output
clear_output()

In [5]:
import os
from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import ipywidgets as widgets
from IPython.display import display, clear_output
clear_output()

In [7]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

print("Descargando el modelo, esto puede tomar unos minutos dependiendo de la conexión...")
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)
print(f"\nModelo descargado en: {model_path}")

Descargando el modelo, esto puede tomar unos minutos dependiendo de la conexión...

Modelo descargado en: /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q4_K_M.gguf


In [8]:
# Instanciamos LlamaCpp con Langchain
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.7,
    max_tokens=256,
    top_p=10,
    verbose=False, # Muestra información del log de C++ debajo
)
clear_output()

In [9]:
# Creamos un prompt adaptado al formato esperado de TinyLlama / LLaMA
template = """<|system|>
Eres un asistente virtual amable y conciso. Responde en español de forma concisa.
<|user|>
{question}
<|assistant|>"""

prompt = PromptTemplate(template=template, input_variables=["question"])
llm_chain = LLMChain(prompt=prompt, llm=llm)

In [ ]:
pregunta = "¿Qué es la inteligencia artificial generativa?"
print(f"Pregunta: {pregunta}\n")
respuesta = llm_chain.invoke(pregunta)
clear_output()
print("--- Respuesta ---")
print(respuesta['text'])

In [ ]:
import gradio as gr

def responder_general(mensaje, historial):
    # Tomar los últimos 3 mensajes del historial
    historial_reciente = historial[-3:]
    contexto = ""
    if historial_reciente:
        contexto = "Historial de conversacion reciente:\n"
        for usr, bot in historial_reciente:
            contexto += f"Usuario: {usr}\nAsistente: {bot}\n\n"
        contexto += "Pregunta actual: "
        
    pregunta_final = contexto + mensaje
    
    # Invoke LLM
    respuesta = llm_chain.invoke(pregunta_final)
    return respuesta['text']

demo = gr.ChatInterface(
    fn=responder_general,
    title="Chatbot General con Memoria",
    description="Este chatbot recuerda los últimos 3 mensajes de la conversación."
)

demo.launch(share=True)